In [44]:
!pip install pyspark

In [56]:
import pandas as pd
import pyspark
from pyspark.ml.feature import MinMaxScaler, StringIndexer, VectorAssembler, PCA
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, month
from pyspark.sql.types import IntegerType
from pyspark.ml.regression import LinearRegression

In [46]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [47]:
spark = SparkSession.builder.getOrCreate()

In [48]:
df_video = spark.read.parquet('/content/videos-comments-tratados.snappy.parquet')
df_video.show()

+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+
|   Video ID|               Title|Published At|Keyword|Likes|Comments|  Views|Interaction|Year|             Comment|Sentiment|Likes Comment|
+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Let's not forget ...|        1|           95|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Here in NZ 50% of...|        0|           19|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|I will forever ac...|        2|          161|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Whenever I go to ...|        0|            8|
|wAZZ-UWGVHI|

In [49]:
df_video = df_video.withColumn("Year", col("Year").cast(IntegerType()))

In [50]:
df_video = df_video.withColumn('Month', month(col('Published At')))
df_video.show()

+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+-----+
|   Video ID|               Title|Published At|Keyword|Likes|Comments|  Views|Interaction|Year|             Comment|Sentiment|Likes Comment|Month|
+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+-----+
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Let's not forget ...|        1|           95|    8|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Here in NZ 50% of...|        0|           19|    8|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|I will forever ac...|        2|          161|    8|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Whenever I go to ...|  

In [51]:
indexador = StringIndexer(inputCol='Keyword', outputCol='Keyword Index')
indexador_keyword = indexador.fit(df_video).transform(df_video)

indexador_keyword.show()

+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+-----+-------------+
|   Video ID|               Title|Published At|Keyword|Likes|Comments|  Views|Interaction|Year|             Comment|Sentiment|Likes Comment|Month|Keyword Index|
+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+-----+-------------+
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Let's not forget ...|        1|           95|    8|         17.0|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Here in NZ 50% of...|        0|           19|    8|         17.0|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|I will forever ac...|        2|          161|    8|         17.0|
|wAZZ-UWGVHI|Apple Pay Is Kill...|

In [52]:
Features = VectorAssembler(
     inputCols = ['Likes','Views','Year','Month','Keyword Index'],
     outputCol = 'Features'
)
feature_campos_df = Features.transform(indexador_keyword)
feature_campos_df.show()

+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+-----+-------------+--------------------+
|   Video ID|               Title|Published At|Keyword|Likes|Comments|  Views|Interaction|Year|             Comment|Sentiment|Likes Comment|Month|Keyword Index|            Features|
+-----------+--------------------+------------+-------+-----+--------+-------+-----------+----+--------------------+---------+-------------+-----+-------------+--------------------+
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Let's not forget ...|        1|           95|    8|         17.0|[3407.0,135612.0,...|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|2022|Here in NZ 50% of...|        0|           19|    8|         17.0|[3407.0,135612.0,...|
|wAZZ-UWGVHI|Apple Pay Is Kill...|  2022-08-23|   tech| 3407|     672| 135612|     139691|

In [53]:
scaler = MinMaxScaler(inputCol='Features', outputCol='Features Normal')
scaler_model = scaler.fit(feature_campos_df)

feature_campos_df = scaler_model.transform(feature_campos_df)
feature_campos_df.show(truncate=False)

+-----------+--------------------------------------------------------------------------------------------------+------------+-------+-----+--------+-------+-----------+----+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+-------------+-----+-------------+-----------------------------------+-------------------------------------------------------------------------------------+
|Video ID   |Title                                                                                             |Published At|Keyword|Likes|Comments|Views  |Interaction|Year|Comment                    

In [54]:
pca = PCA(k=1, inputCol="Features Normal", outputCol="Features PCA")
pca_model = pca.fit(feature_campos_df)
feature_campos_df = pca_model.transform(feature_campos_df)

feature_campos_df.show(truncate=False)

+-----------+--------------------------------------------------------------------------------------------------+------------+-------+-----+--------+-------+-----------+----+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+-------------+-----+-------------+-----------------------------------+-------------------------------------------------------------------------------------+--------------------+
|Video ID   |Title                                                                                             |Published At|Keyword|Likes|Comments|Views  |Interaction|Year|Commen

In [55]:
treino_df, teste_df = feature_campos_df.randomSplit([0.8, 0.2], seed=42)

# Exibe a quantidade de linhas de cada conjunto para verificação
print(f"Quantidade de registros no treino: {treino_df.count()}")
print(f"Quantidade de registros no teste: {teste_df.count()}")

Quantidade de registros no treino: 14789
Quantidade de registros no teste: 3620


In [58]:
regressao_linear = LinearRegression(featuresCol="Features Normal", labelCol="Comments")
lr_model = regressao_linear.fit(treino_df)

avaliar_model = lr_model.evaluate(teste_df)
print(avaliar_model.rootMeanSquaredError)
print(avaliar_model.r2)

25370.3336201662
0.6602413154888491


In [59]:
df_video.write.parquet("videos-preparados-parquet", mode="overwrite")